# ICF Perioperative Analysis
## Patient functioning scores around esofagus surgery

This notebook merges the **ICF notes dataset** (clinical notes with functioning scores extracted as numbers)
with the **esofagus surgical dataset** (one row per operation, with `datok` = date of surgery).

Then it extracts two temporal windows per patient:
- **Pre-op window**: ICF notes recorded in the 30 days **before** surgery
- **Post-op window**: ICF notes recorded in the 40 days **after** surgery

---

## 0. Configuration — change paths here

In [ ]:
# ── EDIT THESE TWO PATHS ──────────────────────────────────────────────────────
# Source data is not included in this repo (patient-level clinical data).
# Place the files locally and point these paths at them to reproduce the analysis.
ICF_PATH    = r"../data/icf.csv"                # ICF notes file (has NOTITIE_DATUM + ICF columns)
SURG_PATH   = r"../data/oesofagus_surgical.csv" # Surgical dataset (has datok)

# Patient ID column name in each file (they can differ)
PATIENT_ID  = "MDN"   # column name in the ICF file
SURG_ID_COL = "upn"   # column name in the surgical file

# Date column in the ICF notes file
NOTE_DATE_COL = "NOTITIE_DATUM"

# Date of surgery column in the surgical file
SURG_DATE_COL = "datok"

# Time windows (days)
PRE_OP_DAYS  = 30    # how many days before surgery to look back
POST_OP_DAYS = 180   # how many days after surgery to look forward (6-month recovery window)
# ─────────────────────────────────────────────────────────────────────────────

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print("Imports OK")

## 2. Load data

In [ ]:
# ── Load ICF notes ────────────────────────────────────────────────────────────
if ICF_PATH.endswith('.csv'):
    icf = pd.read_csv(ICF_PATH, sep=None, engine='python')
else:
    icf = pd.read_excel(ICF_PATH)

print(f"ICF dataset:  {icf.shape[0]:,} rows  x  {icf.shape[1]} columns")
icf.head(3)

In [ ]:
# ── Load surgical dataset ─────────────────────────────────────────────────────
if SURG_PATH.endswith('.csv'):
    surg = pd.read_csv(SURG_PATH, sep=None, engine='python', encoding='utf-8-sig')
else:
    surg = pd.read_excel(SURG_PATH)

print(f"Surgical dataset:  {surg.shape[0]:,} rows  x  {surg.shape[1]} columns")
surg.head(3)

## 3. Parse dates

In [ ]:
def parse_date_col(df, col):
    """Try dayfirst parsing first; fall back to Excel serial if most are NaT."""
    parsed = pd.to_datetime(df[col], dayfirst=True, errors='coerce')
    if parsed.isna().mean() > 0.5:
        parsed = pd.to_datetime(df[col], unit='D', origin='1899-12-30', errors='coerce')
    return parsed

icf[NOTE_DATE_COL]  = parse_date_col(icf,  NOTE_DATE_COL)
surg[SURG_DATE_COL] = parse_date_col(surg, SURG_DATE_COL)

print(f"ICF note dates  — valid: {icf[NOTE_DATE_COL].notna().sum():,}  "
      f"NaT: {icf[NOTE_DATE_COL].isna().sum():,}")
print(f"  range: {icf[NOTE_DATE_COL].min().date()} → {icf[NOTE_DATE_COL].max().date()}")
print()
print(f"Surgery dates   — valid: {surg[SURG_DATE_COL].notna().sum():,}  "
      f"NaT: {surg[SURG_DATE_COL].isna().sum():,}")
print(f"  range: {surg[SURG_DATE_COL].min().date()} → {surg[SURG_DATE_COL].max().date()}")

## 4. Standardise patient ID

In [ ]:
def standardise_id(df, col):
    df = df.copy()
    df[col] = (
        df[col].astype(str)
              .str.strip()
              .str.split('.').str[0]   # remove .0 suffix from floats
              .str.upper()
    )
    return df

icf  = standardise_id(icf,  PATIENT_ID)
surg = standardise_id(surg, SURG_ID_COL)
surg = surg.rename(columns={SURG_ID_COL: PATIENT_ID})  # rename upn → MDN so merge works

# Keep only the columns we need from surg for the merge
surg_slim = surg[[PATIENT_ID, SURG_DATE_COL]].drop_duplicates()

# How many patients overlap?
shared = set(icf[PATIENT_ID].unique()) & set(surg_slim[PATIENT_ID].unique())
print(f"Unique patients in ICF:      {icf[PATIENT_ID].nunique():,}")
print(f"Unique patients in surgical: {surg_slim[PATIENT_ID].nunique():,}")
print(f"Patients in BOTH datasets:   {len(shared):,}")

## 5. Merge ICF notes with surgery date

In [ ]:
merged = icf.merge(surg_slim, on=PATIENT_ID, how='inner')

# Drop rows where either date is missing
before = len(merged)
merged = merged.dropna(subset=[NOTE_DATE_COL, SURG_DATE_COL])
after  = len(merged)
print(f"After merge + dropping missing dates: {after:,} rows  (dropped {before-after:,})")

# Days relative to surgery  (negative = before, positive = after)
merged['days_to_surg'] = (merged[NOTE_DATE_COL] - merged[SURG_DATE_COL]).dt.days

print(f"\ndays_to_surg distribution:")
print(merged['days_to_surg'].describe().to_frame().T.to_string(index=False))

### 5.1 Timeline overview — notes per day relative to surgery

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))

counts = merged['days_to_surg'].value_counts().sort_index()
ax.bar(counts.index, counts.values, width=1, color='steelblue', alpha=0.7)

# Mark the two windows
ax.axvspan(-PRE_OP_DAYS, 0,          alpha=0.15, color='tomato',   label=f'Pre-op window (−{PRE_OP_DAYS} to 0 days)')
ax.axvspan(0,            POST_OP_DAYS, alpha=0.15, color='seagreen', label=f'Post-op window (0 to +{POST_OP_DAYS} days)')
ax.axvline(0, color='black', linewidth=1.5, linestyle='--', label='Surgery day')

ax.set_xlabel('Days relative to surgery (negative = before)')
ax.set_ylabel('Number of ICF notes')
ax.set_title('Distribution of ICF notes around surgery', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Split into pre-op and post-op windows

In [ ]:
# Pre-op: -PRE_OP_DAYS <= days_to_surg < 0
pre_op = merged[
    (merged['days_to_surg'] >= -PRE_OP_DAYS) &
    (merged['days_to_surg'] <  0)
].copy()

# Post-op: 0 < days_to_surg <= POST_OP_DAYS
post_op = merged[
    (merged['days_to_surg'] >  0) &
    (merged['days_to_surg'] <= POST_OP_DAYS)
].copy()

print(f"Pre-op  notes (last {PRE_OP_DAYS} days before surgery): {len(pre_op):,}  "
      f"from {pre_op[PATIENT_ID].nunique():,} patients")
print(f"Post-op notes (first {POST_OP_DAYS} days after surgery): {len(post_op):,}  "
      f"from {post_op[PATIENT_ID].nunique():,} patients")

## 7. Identify ICF variable columns

In [ ]:
# ICF classifier columns all end with '_lvl' (e.g. ENR-B1300_lvl, ATT-B140_lvl)
ICF_COLS = [c for c in merged.columns if c.endswith('_lvl')]

print(f"ICF classifier columns detected: {len(ICF_COLS)}")
print(ICF_COLS)

## 8. Aggregate per patient

Some patients have multiple notes within a window. We summarise using:
- **mean** — average score across all notes in the window
- **last** — the most recent note before surgery (or earliest after)

The mean is safer for sparse data; the last-note captures the most proximal state.

In [ ]:
def aggregate_window(df_window, patient_col, date_col, icf_cols, window_label, keep='last'):
    """
    Aggregate ICF scores per patient within a window.
    keep: 'last'  = most recent note (closest to surgery for pre-op; stable recovery for post-op)
          'first' = earliest note (closest to surgery for post-op)
          'mean'  = average all notes in window
    """
    df_sorted = df_window.sort_values([patient_col, date_col])

    if keep == 'last':
        agg = df_sorted.groupby(patient_col)[icf_cols].last()
    elif keep == 'first':
        agg = df_sorted.groupby(patient_col)[icf_cols].first()
    else:
        agg = df_sorted.groupby(patient_col)[icf_cols].mean()

    agg.columns = [f"{c}_{window_label}" for c in agg.columns]
    return agg.reset_index()


# Pre-op: take the note closest to surgery (= last note when sorted ascending by date)
pre_agg  = aggregate_window(pre_op,  PATIENT_ID, NOTE_DATE_COL, ICF_COLS, 'pre',  keep='last')

# Post-op: take the most recent note within the 180-day window (= stable recovery state)
post_agg = aggregate_window(post_op, PATIENT_ID, NOTE_DATE_COL, ICF_COLS, 'post', keep='last')

print(f"Pre-op  aggregated:  {pre_agg.shape[0]:,} patients  x  {pre_agg.shape[1]-1} ICF scores")
print(f"Post-op aggregated:  {post_agg.shape[0]:,} patients  x  {post_agg.shape[1]-1} ICF scores")

## 9. Build the combined perioperative table

In [ ]:
# Start from the surgical dataset so every operated patient is a row
df_peri = surg_slim.copy()
df_peri = standardise_id(df_peri, PATIENT_ID)

df_peri = df_peri.merge(pre_agg,  on=PATIENT_ID, how='left')
df_peri = df_peri.merge(post_agg, on=PATIENT_ID, how='left')

n_pre  = df_peri[[f"{ICF_COLS[0]}_pre"]].notna().sum().values[0]  if ICF_COLS else 0
n_post = df_peri[[f"{ICF_COLS[0]}_post"]].notna().sum().values[0] if ICF_COLS else 0

print(f"Combined table: {df_peri.shape[0]:,} patients  x  {df_peri.shape[1]} columns")
print(f"  With pre-op ICF data:   {n_pre:,} patients")
print(f"  With post-op ICF data:  {n_post:,} patients")
df_peri.head()

## 10. Coverage: how many patients have ICF data in each window?

In [ ]:
pre_cols  = [c for c in df_peri.columns if c.endswith('_pre')]
post_cols = [c for c in df_peri.columns if c.endswith('_post')]

# A patient is "covered" if at least one ICF variable is non-null
has_pre  = df_peri[pre_cols].notna().any(axis=1)
has_post = df_peri[post_cols].notna().any(axis=1)

N = len(df_peri)
print(f"Total operated patients:          {N:,}")
print(f"With pre-op ICF  (≤{PRE_OP_DAYS}d before):  {has_pre.sum():,}  ({100*has_pre.mean():.1f}%)")
print(f"With post-op ICF (≤{POST_OP_DAYS}d after):   {has_post.sum():,}  ({100*has_post.mean():.1f}%)")
print(f"With BOTH windows:                {(has_pre & has_post).sum():,}  "
      f"({100*(has_pre & has_post).mean():.1f}%)")

## 11. Descriptive statistics — pre-op vs post-op

In [ ]:
rows = []
for col in ICF_COLS:
    pre_vals  = df_peri[f"{col}_pre"].dropna()
    post_vals = df_peri[f"{col}_post"].dropna()

    rows.append({
        'ICF variable': col,
        'N pre-op':  len(pre_vals),
        'Mean pre':  round(pre_vals.mean(),  2) if len(pre_vals)  > 0 else np.nan,
        'SD pre':    round(pre_vals.std(),   2) if len(pre_vals)  > 0 else np.nan,
        'Median pre': round(pre_vals.median(),2) if len(pre_vals) > 0 else np.nan,
        'N post-op': len(post_vals),
        'Mean post': round(post_vals.mean(), 2) if len(post_vals) > 0 else np.nan,
        'SD post':   round(post_vals.std(),  2) if len(post_vals) > 0 else np.nan,
        'Median post': round(post_vals.median(),2) if len(post_vals) > 0 else np.nan,
    })

desc = pd.DataFrame(rows)
print(f"Descriptive table — {len(desc)} ICF variables")
desc

## 12. Change scores: Δ = post − pre

In [ ]:
# Compute delta only for patients who have BOTH windows
delta_cols = []
for col in ICF_COLS:
    pre_c  = f"{col}_pre"
    post_c = f"{col}_post"
    if pre_c in df_peri.columns and post_c in df_peri.columns:
        df_peri[f"{col}_delta"] = df_peri[post_c] - df_peri[pre_c]
        delta_cols.append(f"{col}_delta")

print(f"Delta columns created: {len(delta_cols)}")

delta_summary = df_peri[delta_cols].describe().loc[['count','mean','std','min','50%','max']].T
delta_summary.index = [c.replace('_delta','') for c in delta_cols]
delta_summary.columns = ['N pairs', 'Mean Δ', 'SD Δ', 'Min Δ', 'Median Δ', 'Max Δ']
delta_summary = delta_summary.sort_values('Mean Δ')
print("\nChange scores (post − pre), sorted by mean change:")
delta_summary

## 13. Visualisation — pre vs post mean scores

In [ ]:
# Bar chart: mean pre vs mean post for each ICF variable
plot_data = desc.set_index('ICF variable')[['Mean pre', 'Mean post']].dropna(how='all')

if len(plot_data) == 0:
    print("No data to plot — check that ICF columns are correctly identified.")
else:
    n_vars = len(plot_data)
    fig, ax = plt.subplots(figsize=(max(14, n_vars * 0.6), 5))

    x = np.arange(n_vars)
    w = 0.38
    ax.bar(x - w/2, plot_data['Mean pre'],  width=w, color='tomato',   label=f'Pre-op (≤{PRE_OP_DAYS}d before)',  alpha=0.85)
    ax.bar(x + w/2, plot_data['Mean post'], width=w, color='seagreen', label=f'Post-op (≤{POST_OP_DAYS}d after)', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_data.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Mean ICF level')
    ax.set_title('Mean ICF scores — pre-op vs post-op', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:

from matplotlib.patches import Patch

ICF_LABELS = {
    'INS-B455_lvl':      'Exercise tolerance',
    'HRN-B230_lvl':      'Hearing',
    'MAE-D465_lvl':      'Mobility (equipment)',
    'ADM-B440_lvl':      'Respiration',
    'ENR-B1300_lvl':     'Energy & drive',
    'BER-D840-D859_lvl': 'Work & employment',
    'SOP-B280_lvl':      'Pain',
    'FAC-D540_lvl':      'Self-care',
    'MBW-B530_lvl':      'Body weight',
    'FML-D760_lvl':      'Family relations',
    'CBP-D410_lvl':      'Body positioning',
    'HSP-D240_lvl':      'Handling stress',
    'STM-B152_lvl':      'Emotional functions',
    'SLP-B134_lvl':      'Sleep',
    'ATT-B140_lvl':      'Attention',
    'HLC-B164_lvl':      'Higher cognition',
    'ETN-D550_lvl':      'Eating',
}

# Threshold: domains with fewer than N_MIN paired observations are
# "rarely documented" — shown with hatching to signal limited reliability
# while still displaying the direction of change
N_MIN = 50

if len(delta_summary) > 0:
    delta_labeled = delta_summary.copy()
    delta_labeled.index = [ICF_LABELS.get(idx, idx) for idx in delta_labeled.index]

    fig, ax = plt.subplots(figsize=(11, max(7, len(delta_labeled) * 0.58)))

    for domain, row in delta_labeled.iterrows():
        val    = row['Mean Δ']
        n      = int(row['N pairs'])
        is_rare = n < N_MIN

        color = 'seagreen' if val >= 0 else 'tomato'
        alpha  = 0.45 if is_rare else 0.85
        hatch  = '//' if is_rare else None
        edge   = '#555555' if is_rare else 'white'

        ax.barh(domain, val,
                color=color, alpha=alpha,
                edgecolor=edge, linewidth=0.8,
                hatch=hatch)

        # n= label at the tip of each bar
        pad = 0.018
        x_lbl = val + pad if val >= 0 else val - pad
        ha     = 'left'   if val >= 0 else 'right'
        ax.text(x_lbl, domain, f'n={n}',
                va='center', ha=ha, fontsize=8.5, color='dimgray')

    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel(
        'Mean change (post − pre)  ·  positive = improvement  (scale: 0 = worst, 4 = best)',
        fontsize=11)
    ax.set_title(
        'ICF change scores after esophagectomy\n'
        'Green = improvement (Δ > 0);  Red = worsening (Δ < 0)',
        fontweight='bold', fontsize=13)
    ax.tick_params(axis='y', labelsize=11)
    ax.tick_params(axis='x', labelsize=10)

    legend_elements = [
        Patch(facecolor='grey', alpha=0.85,
              label=f'Well-documented  (n ≥ {N_MIN} paired patients)'),
        Patch(facecolor='grey', alpha=0.45, hatch='//', edgecolor='#555555',
              label=f'Rarely documented  (n < {N_MIN} paired patients)'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10,
              framealpha=0.9)

    plt.tight_layout()
    # Output figure path — relative to this notebook's location
    plt.savefig(
        '../figures/ICF_change.png',
        dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: ICF_change.png at 300 DPI')


In [ ]:
# Boxplots for each ICF variable — pre vs post distributions
# Shows more than just the mean — useful for skewed scores
n_show = min(len(ICF_COLS), 12)   # cap at 12 to keep plot readable

if n_show > 0:
    fig, axes = plt.subplots(2, n_show, figsize=(n_show * 1.4, 7), sharey=False)
    if n_show == 1:
        axes = axes.reshape(2, 1)

    for i, col in enumerate(ICF_COLS[:n_show]):
        pre_vals  = df_peri[f"{col}_pre"].dropna()
        post_vals = df_peri[f"{col}_post"].dropna()
        short = col.split('_')[0] if '_' in col else col

        for ax, vals, color, label in [
            (axes[0, i], pre_vals,  'tomato',   'pre'),
            (axes[1, i], post_vals, 'seagreen', 'post'),
        ]:
            if len(vals) > 0:
                ax.boxplot(vals, widths=0.6, patch_artist=True,
                           boxprops=dict(facecolor=color, alpha=0.6),
                           medianprops=dict(color='black', linewidth=2),
                           flierprops=dict(marker='.', markersize=3, alpha=0.4))
            ax.set_title(short, fontsize=7)
            ax.set_xticks([])
            if i == 0:
                ax.set_ylabel(label, fontsize=8)

    fig.suptitle(f'ICF score distributions — pre-op (top) vs post-op (bottom)', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 13b. Temporal distribution — how ICF scores evolve day by day around surgery

Uses every note in `merged` (not just the windowed aggregates) so you see the full trajectory.
The x-axis is days relative to surgery; the shaded band is ±1 SD across patients.

In [ ]:
# ── Day-by-day mean ± SD line plot for each ICF variable ─────────────────────
# Only show the combined window: -PRE_OP_DAYS to +POST_OP_DAYS
window = merged[
    (merged['days_to_surg'] >= -PRE_OP_DAYS) &
    (merged['days_to_surg'] <= POST_OP_DAYS)
].copy()

# How many variables to plot (cap to avoid giant figure)
n_plot = min(len(ICF_COLS), 16)
ncols  = 4
nrows  = int(np.ceil(n_plot / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.2), sharey=False)
axes = axes.flatten()

for i, col in enumerate(ICF_COLS[:n_plot]):
    ax = axes[i]
    grp = window.groupby('days_to_surg')[col]
    day_mean = grp.mean()
    day_sd   = grp.std()
    day_n    = grp.count()

    # Only keep days with ≥ 3 observations so the line is meaningful
    mask = day_n >= 3
    x    = day_mean.index[mask]
    y    = day_mean.values[mask]
    sd   = day_sd.values[mask]

    ax.fill_between(x, y - sd, y + sd, alpha=0.18, color='steelblue')
    ax.plot(x, y, color='steelblue', linewidth=1.6)
    ax.axvline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.6)
    ax.axvspan(-PRE_OP_DAYS, 0,           alpha=0.07, color='tomato')
    ax.axvspan(0,             POST_OP_DAYS, alpha=0.07, color='seagreen')

    short = col.replace('_lvl', '').replace('_', ' ')
    ax.set_title(short, fontsize=8, fontweight='bold')
    ax.set_xlabel('Days to surgery', fontsize=7)
    ax.set_ylabel('Mean level', fontsize=7)
    ax.tick_params(labelsize=7)

# Hide unused subplots
for j in range(n_plot, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('ICF scores over time relative to surgery  (mean ± SD,  red=pre  green=post)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap: mean ICF level per variable × weekly time bin ───────────────────
# Bins: each week before/after surgery
bin_edges  = list(range(-PRE_OP_DAYS, POST_OP_DAYS + 1, 7))
bin_labels = [f"{b+1} to {b+7}" if b >= 0 else f"{b} to {b+6}"
              for b in bin_edges[:-1]]

window['time_bin'] = pd.cut(window['days_to_surg'],
                             bins=bin_edges, labels=bin_labels, right=True)

heatmap_data = (
    window.groupby('time_bin', observed=True)[ICF_COLS]
          .mean()
          .T
)

# Drop columns (time bins) that are entirely NaN
heatmap_data = heatmap_data.dropna(axis=1, how='all')

if heatmap_data.shape[1] > 0:
    fig, ax = plt.subplots(figsize=(max(12, heatmap_data.shape[1] * 1.2),
                                    max(5, len(ICF_COLS) * 0.45)))

    sns.heatmap(heatmap_data, ax=ax, cmap='RdYlGn_r', annot=True, fmt='.1f',
                linewidths=0.4, linecolor='white',
                cbar_kws={'label': 'Mean ICF level'},
                annot_kws={'size': 7})

    # Find which column corresponds to the first post-op bin and draw a line
    n_pre_bins = sum(1 for l in heatmap_data.columns
                     if l.startswith('-') or l.startswith('−'))
    ax.axvline(n_pre_bins, color='black', linewidth=2.5)

    short_idx = [c.replace('_lvl', '') for c in heatmap_data.index]
    ax.set_yticklabels(short_idx, rotation=0, fontsize=8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
    ax.set_title('Mean ICF level per variable per week  (black line = surgery)',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to build heatmap — check that notes fall within the time windows.")

In [ ]:
# ── KDE distributions: pre-op vs post-op per ICF variable ────────────────────
# Shows the full shape of the score distribution, not just the mean
n_plot = min(len(ICF_COLS), 16)
ncols  = 4
nrows  = int(np.ceil(n_plot / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3), sharey=False)
axes = axes.flatten()

for i, col in enumerate(ICF_COLS[:n_plot]):
    ax = axes[i]

    pre_vals  = pre_op[col].dropna()
    post_vals = post_op[col].dropna()

    if len(pre_vals) >= 5:
        pre_vals.plot.kde(ax=ax, color='tomato',   linewidth=2,
                          label=f'Pre-op  (n={len(pre_vals)})')
        ax.axvline(pre_vals.median(),  color='tomato',   linestyle='--', linewidth=1, alpha=0.7)

    if len(post_vals) >= 5:
        post_vals.plot.kde(ax=ax, color='seagreen', linewidth=2,
                           label=f'Post-op (n={len(post_vals)})')
        ax.axvline(post_vals.median(), color='seagreen', linestyle='--', linewidth=1, alpha=0.7)

    short = col.replace('_lvl', '').replace('_', ' ')
    ax.set_title(short, fontsize=8, fontweight='bold')
    ax.set_xlabel('ICF level', fontsize=7)
    ax.set_ylabel('Density', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6)

for j in range(n_plot, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('ICF score distributions — pre-op (red) vs post-op (green)\n'
             'Dashed lines = median', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 14. Paired significance test (Wilcoxon signed-rank)

For each ICF variable, we test whether the pre-op and post-op scores differ significantly.
We use the **Wilcoxon signed-rank test** (non-parametric, appropriate for small samples and ordinal ICF levels).

In [ ]:
from scipy.stats import wilcoxon

test_rows = []
for col in ICF_COLS:
    pair = df_peri[[f"{col}_pre", f"{col}_post"]].dropna()
    n = len(pair)
    if n < 10:
        test_rows.append({'ICF variable': col, 'N pairs': n,
                          'Mean Δ': np.nan, 'p-value': np.nan, 'Significant': 'too few'})
        continue

    pre_v  = pair[f"{col}_pre"].values
    post_v = pair[f"{col}_post"].values

    try:
        stat, p = wilcoxon(pre_v, post_v, zero_method='wilcox')
    except ValueError:
        p = np.nan  # all differences are zero

    mean_delta = (post_v - pre_v).mean()
    test_rows.append({
        'ICF variable': col,
        'N pairs': n,
        'Mean Δ': round(mean_delta, 3),
        'p-value': round(p, 4) if not np.isnan(p) else np.nan,
        'Significant': '✓' if (not np.isnan(p) and p < 0.05) else '–',
    })

test_df = pd.DataFrame(test_rows).sort_values('p-value')
print("Wilcoxon signed-rank test — pre vs post surgery:")
test_df

## 15. Export results

In [ ]:
# Main combined table
df_peri.to_csv('icf_perioperative.csv', index=False)
print(f"Saved icf_perioperative.csv  —  {df_peri.shape[0]:,} patients  x  {df_peri.shape[1]} columns")

# Descriptive stats
desc.to_csv('icf_descriptive_pre_post.csv', index=False)
print(f"Saved icf_descriptive_pre_post.csv")

# Statistical tests
test_df.to_csv('icf_wilcoxon_tests.csv', index=False)
print(f"Saved icf_wilcoxon_tests.csv")